# RAG end-to-end on Cloud SQL — embeddings × chunking, benchmarked (psycopg2 edition)

**New to this? Start here.** RAG = *Retrieval-Augmented Generation*. To answer a question about your documents you: **(1)** cut documents into pieces (*chunking*), **(2)** turn each piece into a vector of numbers that captures its meaning (*embedding*), **(3)** store those vectors in Postgres (*pgvector*), **(4)** for a question, find the closest pieces (*vector search*), **(5)** let Gemini write the answer from those pieces.

The quality depends on **which embedding model** and **which chunking method** you pick — and there's no universal best, it depends on *your* data. So this notebook tries several of each, **measures them**, and tells you the winner.

Same connection style as your `vector_search_psycopg2` notebook (`DB_CONFIG` + `psycopg2` + `google-genai` Vertex). Run cells top-to-bottom.

| Step | What happens |
|---|---|
| 1–3 | Install, configure, **connect to Cloud SQL** |
| 4 | Enable pgvector |
| 5 | Vertex embedding function (works for any model) |
| 6 | **Chunking** — three techniques, side by side |
| 7 | Your documents + a small *gold set* (for scoring) |
| 8 | **Ingest** — chunk + embed + store, once per combo |
| 9 | Vector search |
| 10 | **Benchmark** — recall@k & MRR leaderboard → the best combo |
| 11 | Answer questions with Gemini using the winner |
| 12 | Cleanup + troubleshooting |

## Step 1 — Install

In [ ]:
%pip install -q google-genai psycopg2-binary tiktoken pandas
print("Installed. If pip shows a 'restart kernel' notice, restart the kernel now.")

## Step 2 — Configuration

**Copy `DB_CONFIG` from your working `vector_search_psycopg2` / `sql_query_expert` notebook.** If you connect through the Cloud SQL Auth Proxy, keep `host="localhost"` and make sure the proxy is running.

`EMBEDDERS` and `CHUNKERS` are the methods we compare — add or remove freely.

In [ ]:
# ---------- Google Cloud / Vertex AI ----------
PROJECT_ID = "your-gcp-project-id"     # CHANGE
LOCATION   = "us-central1"
CHAT_MODEL = "gemini-2.5-flash"

# ---------- PostgreSQL / Cloud SQL (copy from your other notebook) ----------
DB_CONFIG = {
    "host":     "localhost",           # CHANGE if needed
    "port":     5432,
    "dbname":   "your_database",       # CHANGE
    "user":     "postgres",            # CHANGE
    "password": "your_password",       # CHANGE
}

# ---------- Methods to compare ----------
# Embedding models (all Vertex / google-genai). Dimension is auto-detected, so you
# don't have to know it — just list the model names.
EMBEDDERS = [
    "text-embedding-004",     # the "text embedding 4" you mentioned (768-dim)
    "text-embedding-005",     # newer general-purpose (768-dim)
    "gemini-embedding-001",   # highest quality, larger (3072-dim)
]

# Chunking methods are defined in Step 6; we reference them by name here.
CHUNKERS = ["fixed_400", "sentence", "token_256"]

TOP_K = 3   # retrieve/score the top-3 pieces per question
print("Config loaded.")

## Step 3 — Connect to Cloud SQL (same pattern as your notebook)

In [ ]:
import psycopg2
import psycopg2.extras

def get_conn():
    return psycopg2.connect(**DB_CONFIG)

try:
    with get_conn() as conn, conn.cursor() as cur:
        cur.execute("SELECT current_database(), version();")
        dbname, version = cur.fetchone()
    print(f"Connected to '{dbname}'")
    print("Server:", version.split(',')[0])
except Exception as e:
    raise SystemExit(f"Could not connect: {e}\n  -> Check DB_CONFIG. If host is 'localhost', is the Cloud SQL Auth Proxy running?")

## Step 4 — Enable pgvector

One-time. Cloud SQL for PostgreSQL supports the `vector` extension.

In [ ]:
with get_conn() as conn, conn.cursor() as cur:
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    conn.commit()
print("pgvector ready.")

## Step 5 — Vertex embedding function

**The golden rule:** documents and questions must be embedded by the *same* model. We pass `task_type` so the model knows whether it's embedding a *document* (`RETRIEVAL_DOCUMENT`) or a *search query* (`RETRIEVAL_QUERY`) — this improves results. The function batches and falls back to one-at-a-time if a model rejects batches.

In [ ]:
from google import genai
from google.genai.types import EmbedContentConfig

genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

def embed_texts(model: str, texts: list[str], task_type: str) -> list[list[float]]:
    out: list[list[float]] = []
    for i in range(0, len(texts), 50):
        batch = texts[i:i + 50]
        cfg = EmbedContentConfig(task_type=task_type)
        try:
            resp = genai_client.models.embed_content(model=model, contents=batch, config=cfg)
            out.extend([e.values for e in resp.embeddings])
        except Exception:
            for t in batch:  # some models only accept one input per call
                resp = genai_client.models.embed_content(model=model, contents=[t], config=cfg)
                out.append(resp.embeddings[0].values)
    return out

_probe = embed_texts("text-embedding-004", ["hello"], "RETRIEVAL_QUERY")[0]
print(f"Vertex works — text-embedding-004 returns {len(_probe)} dims.")

## Step 6 — Chunking methods (three techniques)

Each function turns one document into a list of text pieces:

- **`fixed_400`** — a sliding window of ~400 characters with overlap. Simplest; predictable size.
- **`sentence`** — packs whole sentences together up to a size. Respects natural boundaries → cleaner meaning.
- **`token_256`** — windows measured in *model tokens* (not characters). Best for staying within model limits.

Overlap means neighbouring pieces share some text, so an answer that straddles a boundary isn't lost.

In [ ]:
import re
import tiktoken

_enc = tiktoken.get_encoding("cl100k_base")
_SENT = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9])")

def chunk_fixed(text, size=400, overlap=60):
    text = text.strip()
    step = max(1, size - overlap)
    return [text[i:i + size] for i in range(0, len(text), step) if text[i:i + size].strip()]

def chunk_sentence(text, size=500):
    sents = [s.strip() for s in _SENT.split(text.strip()) if s.strip()]
    chunks, buf = [], ""
    for s in sents:
        if buf and len(buf) + len(s) > size:
            chunks.append(buf); buf = s
        else:
            buf = f"{buf} {s}".strip()
    if buf:
        chunks.append(buf)
    return chunks

def chunk_tokens(text, size=256, overlap=40):
    toks = _enc.encode(text)
    step = max(1, size - overlap)
    return [_enc.decode(toks[i:i + size]) for i in range(0, len(toks), step)]

CHUNKER_FUNCS = {
    "fixed_400": lambda t: chunk_fixed(t, 400, 60),
    "sentence":  lambda t: chunk_sentence(t, 500),
    "token_256": lambda t: chunk_tokens(t, 256, 40),
}

demo = "Acme Foods quoted 3.20 USD per case for romaine. Payment terms were Net-30. Delivery was twice weekly."
for name, fn in CHUNKER_FUNCS.items():
    print(name, "->", fn(demo))

## Step 7 — Your documents + gold set

`docs` = the text to index. `gold` = evaluation pairs (`question → the document that should answer it`). The gold set is how we score "which method is best." **Replace these samples with your own RFP/bid text and real questions** — the more realistic the gold set, the more trustworthy the result.

In [ ]:
docs = {
    "acme-produce-2024": (
        "Acme Foods bid on the produce category for Riverside USD on 2024-05-01. "
        "Romaine lettuce: 3.20 USD/case. Spinach: 4.10 USD/case. Carrots: 2.05 USD/case. "
        "Payment terms: Net-30. Delivery: twice weekly."
    ),
    "globex-dairy-2024": (
        "Globex Dairy responded to the dairy RFP for Riverside USD. "
        "Whole milk: 1.85 USD/gallon. Cheddar block: 3.40 USD/lb. Terms: Net-45. "
        "They offered a 2% early-payment discount and requested a 12-month contract."
    ),
    "bid-summary-2024": (
        "Bid BID-2024-089 for Riverside USD: eight suppliers solicited, five responded "
        "(62.5% response rate). Internal due 2024-05-20, customer due 2024-05-25. Status: Active."
    ),
}

gold = {
    "What did Acme quote for romaine?": "acme-produce-2024",
    "Which supplier offered an early payment discount?": "globex-dairy-2024",
    "What were Globex's payment terms?": "globex-dairy-2024",
    "What was the supplier response rate?": "bid-summary-2024",
    "When is the customer due date?": "bid-summary-2024",
}
print(f"{len(docs)} documents, {len(gold)} evaluation questions.")

## Step 8 — Ingest: chunk + embed + store (one table per combo)

For each (embedding model × chunking method) we build a separate table so results are comparable. The vector dimension is **auto-detected** from the model, so mixing 768-dim and 3072-dim models just works.

In [ ]:
from psycopg2.extras import execute_values

def table_name(model, chunker):
    slug = re.sub(r"[^a-z0-9]+", "_", f"{model}_{chunker}".lower()).strip("_")
    return f"rag_bench_{slug}"[:60]

def ingest(model, chunker):
    """Chunk every doc with `chunker`, embed with `model`, store. Returns (table, dim, n_chunks)."""
    fn = CHUNKER_FUNCS[chunker]
    rows = []                       # (doc_id, chunk_index, content)
    for doc_id, text in docs.items():
        for idx, piece in enumerate(fn(text)):
            rows.append((doc_id, idx, piece))

    vectors = embed_texts(model, [r[2] for r in rows], "RETRIEVAL_DOCUMENT")
    dim = len(vectors[0])
    table = table_name(model, chunker)

    with get_conn() as conn, conn.cursor() as cur:
        cur.execute(f"DROP TABLE IF EXISTS {table}")
        cur.execute(f"""
            CREATE TABLE {table} (
                id serial PRIMARY KEY,
                doc_id text, chunk_index int, content text, embedding vector({dim})
            )
        """)
        execute_values(
            cur,
            f"INSERT INTO {table} (doc_id, chunk_index, content, embedding) VALUES %s",
            [(rows[i][0], rows[i][1], rows[i][2], str(vectors[i])) for i in range(len(rows))],
            template="(%s, %s, %s, %s::vector)",
        )
        cur.execute(f"CREATE INDEX ON {table} USING hnsw (embedding vector_cosine_ops)")
        conn.commit()
    return table, dim, len(rows)

print("ingest() ready.")

## Step 9 — Vector search

`<=>` is pgvector's cosine distance (0 = identical meaning). We embed the question with the **same model** and return the closest pieces.

In [ ]:
def search(table, model, question, top_k=TOP_K):
    qv = embed_texts(model, [question], "RETRIEVAL_QUERY")[0]
    with get_conn() as conn, conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        cur.execute(f"""
            SELECT doc_id, content, embedding <=> %s::vector AS distance
            FROM {table} ORDER BY distance LIMIT %s
        """, (str(qv), top_k))
        return [dict(r) for r in cur.fetchall()]

print("search() ready.")

## Step 10 — Benchmark every combo → leaderboard

For each combo we ingest, then score the gold questions:
- **recall@k** — how often the correct document is in the top-k.
- **MRR** — how *high* it ranks (1.0 = always first).

Combos whose model isn't available are skipped and reported (they won't stop the run).

In [ ]:
def score(table, model):
    recall_hits, rr_sum = 0, 0.0
    for query, expected in gold.items():
        ranked = [h["doc_id"] for h in search(table, model, query, TOP_K)]
        if expected in ranked:
            recall_hits += 1
            rr_sum += 1.0 / (ranked.index(expected) + 1)
    n = len(gold)
    return recall_hits / n, rr_sum / n

results = []
for model in EMBEDDERS:
    for chunker in CHUNKERS:
        label = f"{model} | {chunker}"
        try:
            table, dim, n = ingest(model, chunker)
            recall, mrr = score(table, model)
            results.append({"label": label, "recall": recall, "mrr": mrr,
                            "table": table, "dim": dim, "ok": True})
            print(f"✓ {label:<42} dim={dim:<5} recall@{TOP_K}={recall:.2f} mrr={mrr:.2f}")
        except Exception as e:
            results.append({"label": label, "ok": False, "error": f"{type(e).__name__}: {e}"})
            print(f"✗ {label:<42} skipped ({str(e)[:50]})")

In [ ]:
ok = sorted([r for r in results if r["ok"]], key=lambda r: (r["mrr"], r["recall"]), reverse=True)

print(f"{'rank':<5}{'combo':<44}{'recall@'+str(TOP_K):<12}{'mrr':<6}")
print("-" * 68)
for i, r in enumerate(ok, 1):
    print(f"{i:<5}{r['label']:<44}{r['recall']:<12.2f}{r['mrr']:<6.2f}")

if ok:
    best = ok[0]
    print(f"\n⭐ Best for your data: {best['label']}  (table: {best['table']})")
    BEST_MODEL = best["label"].split(" | ")[0]
    BEST_TABLE = best["table"]

## Step 11 — Answer questions with Gemini (using the winning combo)

`ask()` retrieves from the best table and lets Gemini answer from those pieces only.

In [ ]:
def ask(question, top_k=TOP_K):
    hits = search(BEST_TABLE, BEST_MODEL, question, top_k)
    context = "\n\n---\n\n".join(h["content"] for h in hits)
    prompt = (
        "Answer the question using ONLY the context below. "
        "If the context doesn't contain the answer, say you don't know.\n\n"
        f"CONTEXT:\n{context}\n\nQUESTION: {question}"
    )
    resp = genai_client.models.generate_content(model=CHAT_MODEL, contents=prompt)
    return resp.text

print(ask("What did Acme quote for romaine, and what were the payment terms?"))

## Step 12 — Notes & cleanup

**Reading the leaderboard**
- Pick the top combo; that's your best `model + chunking` for this data.
- With only 5 gold questions the scores are coarse — add 20–50 real questions for a trustworthy result.
- On ties, prefer the smaller/cheaper model (768-dim is cheaper to store/search than 3072-dim).

**Going further (better results)**
- Add **hybrid search** (vector + keyword) and a **reranker** — already implemented in `services/rag/` (`docs/rag_pgvector_guide.md`).
- Keep the winning table for production instead of dropping it.

The cleanup below drops the temporary `rag_bench_*` tables. Skip it if you want to keep the winner.

In [ ]:
with get_conn() as conn, conn.cursor() as cur:
    for r in results:
        if r.get("table"):
            cur.execute(f"DROP TABLE IF EXISTS {r['table']}")
    conn.commit()
print("Temporary benchmark tables dropped.")

## Troubleshooting

| Symptom | Fix |
|---|---|
| `could not connect to server` | DB_CONFIG wrong, or Cloud SQL Auth Proxy not running (Step 2/3) |
| `403 PERMISSION_DENIED` from Vertex | `gcloud auth application-default login` + enable the Vertex AI API |
| `type "vector" does not exist` | Re-run Step 4 (enable pgvector) against the right database |
| A model is skipped in Step 10 | That model isn't enabled in your project/region — remove it from `EMBEDDERS` |
| Search slow on a big table | the HNSW index is created in Step 8; for very large tables tune it later |
| Good chunks, weak answers | raise `TOP_K`, or improve the prompt in Step 11 |